# W05A · CLIP 의미 검색과 누적 웹 앱
2026-09-28 · 생성형 인공지능 · 주교재 3장

B는 이미지·문장 임베딩 검색입니다. CLIP 가중치 약 605 MB를 최초 다운로드하고 캐시를 재사용합니다. A와 독립 실행합니다. 지난 W04A를 마치지 않았어도 새 사본에서 시작할 수 있습니다.
CPU 기본, API 키·개인 파일 업로드 불필요. 패키지 **0.1.6**, 고정 태그 **2026-fall-w05a**.
[강의노트](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/course/notion/w05a/w05a-image-representations.md) · [기록지](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/course/notion/w05a/w05a-workbook.md) · [전체 API·소스 바로가기](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/W05A-API.md)

빈칸은 별도 연습 변수에 적습니다. None을 남겨도 기본 관찰을 막지 않습니다.
기본 코드의 결과를 먼저 예측하고 실행한 다음, 마지막 점검 셀까지 확인하세요.


## 0. 환경 준비
설치는 패키지와 수업 체크포인트·사진을 함께 준비합니다. 모델이나 데이터는 필요한 준비 셀에서 받습니다.
이미 import한 패키지를 교체하려면 런타임을 재시작하세요. 패키지 버전과 실제 Python 버전을 출력합니다.


In [ ]:
import sys, subprocess, os
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
PUBLIC_REF = '2026-fall-w05a'
try:
    ready = version('luna-genai') == '0.1.6' and version('transformers') == '5.15.1' and version('gradio') == '6.26.0'
except PackageNotFoundError:
    ready = False
if not ready:
    if any(x in sys.modules for x in ('luna_genai','torch','transformers','gradio')):
        raise RuntimeError('런타임을 재시작하고 첫 설치 셀부터 실행하세요.')
    package = str(Path('src').resolve()) if Path('src/pyproject.toml').exists() else f'git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src'
    subprocess.run([sys.executable,'-m','pip','install','-q',package],check=True)
print('Python',sys.version.split()[0],'luna-genai',version('luna-genai'),'transformers',version('transformers'))
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
torch.set_num_threads(2)


## 1. 사진 모음과 출처
[`load_gallery`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L36)는 체크섬을 검증한 40장과 메타데이터를 DataFrame으로 반환합니다. 사진은 패키지 설치 때 자동 준비됩니다.
ID, category, title, author, license, source, path가 있으며 path는 PIL/Gradio용 로컬 파일입니다. category와 파일명은 CLIP 입력에 사용하지 않습니다.
[사진별 저자·이용 조건](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/GALLERY-CREDITS.md)을 확인하세요. 각 사진의 원본 조건이 유지됩니다.


In [ ]:
from luna_genai.semantic_search import (load_gallery,CLIPSearch,unit_rows,precision_at_k,search_view,CLIP_ID,CLIP_REVISION)
gallery=load_gallery()
display(gallery[['id','category','license','source']])
from PIL import Image
fig,axes=plt.subplots(5,8,figsize=(16,10))
for ax,row in zip(axes.flat,gallery.itertuples()):
    ax.imshow(Image.open(row.path));ax.set_title(row.id);ax.axis('off')
display(fig);plt.close(fig)


### B1 · 검색 기준을 먼저 정하기
목표: 평가 기준 사후 변경 방지. 눈 위의 고양이와 빨간 자동차에 해당하는 사진 ID를 결과 검색 전에 적으세요. 힌트: 위 전체 사진 목록. 자가 점검: 대상과 속성 조건을 모두 확인합니다.


## 2. CLIP 모델 준비와 이미지 인덱스
[`CLIPSearch`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L65) 생성자는 고정 revision의 실제 모델을 CPU에 불러옵니다. 첫 다운로드 약605 MB, API 키 불필요입니다.
[`CLIPSearch.index_images`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L137)는 사진 픽셀을 정규화된 `(40,512)` 임베딩으로 만들어 캐시합니다. 이름은 index이지만 가중치 학습은 없습니다.
[`CLIPSearch.encode_images`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L106)는 모델의 크기 조정·중앙 자르기·정규화를 적용합니다. [`CLIPSearch.encode_texts`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L125)는 짧은 문장 목록을 `(N,512)`로 변환합니다. 영어 권장, 최대77토큰입니다.
이 버전의 feature 함수는 `.pooler_output`에서 투영된 벡터를 꺼냅니다. [현재 버전 CLIP API](https://huggingface.co/docs/transformers/v5.15.1/en/model_doc/clip).


In [ ]:
lab=CLIPSearch(gallery)
image_vectors=lab.index_images()
print('모델:',CLIP_ID,'revision:',CLIP_REVISION)
print('새 모델 파일:',lab.model_downloads or '없음: 모델 캐시 재사용')
print('사진 임베딩:',image_vectors.shape,'임베딩 캐시 재사용:',lab.cache_reused)
print('앞 5장 벡터 길이:',image_vectors[:5].norm(dim=1))
print('이미지 내부 / 텍스트 내부 / 투영 차원:',lab.model.config.vision_config.hidden_size,
      lab.model.config.text_config.hidden_size,lab.model.config.projection_dim)


## 3. 정규화 축과 코사인 수계산
[`unit_rows`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L23)는 각 행 벡터를 L2 길이로 나눕니다. finite float `(N,d)`가 입력이고 같은 shape이 출력입니다. 0벡터는 오류입니다.
설명용 벡터 `(3,4)`와 `(0,2)`의 내적은8, 정규화한 내적은0.8입니다. 실제 CLIP 점수와 구별합니다.


In [ ]:
toy_vectors=torch.tensor([[3.,4.],[0.,2.]])
normalized=unit_rows(toy_vectors)
print('단위 벡터:',normalized,'벡터 길이:',normalized.norm(dim=1))
print('정규화 전 내적:',toy_vectors[0]@toy_vectors[1])
print('cosine:',normalized[0]@normalized[1])


### B2 · 정규화 디버깅
목표: 표본 축·특징 축 구별. `(40,512)`에서 `norm(dim=0)`으로 나눈 코드가 왜 잘못되는지 설명하고, 연습 변수에 올바른 행별 정규화 값을 넣으세요. 힌트: 길이 하나는 사진 하나에 대응. 자가 점검: 결과40개의 행 길이가1입니다.


In [ ]:
my_normalized=None  # 연습: toy_vectors를 각 행 길이로 나누기
if my_normalized is not None: print('내 행 길이:',my_normalized.norm(dim=1))


## 4. 실제 3×3 짝맞추기 행렬
사진 P01·P09·P17과 dog·cat·pizza 문장을 비교합니다. 사진이 행, 문장이 열입니다.
스케일과 softmax는 후보 집합에서 상대 점수를 만들며 정답 확률의 보정이 아닙니다. 이 실습은 사전학습 CLIP을 추론만 합니다.


In [ ]:
texts=['a photo of a dog','a photo of a cat','a photo of pizza']
text_vectors=lab.encode_texts(texts)
similarities=image_vectors[[0,8,16]]@text_vectors.T
display(pd.DataFrame(similarities.numpy(),index=['P01 dog','P09 cat','P17 pizza'],columns=texts))
scale=float(lab.model.logit_scale.exp().detach())
relative=torch.softmax(similarities*scale,dim=1)
print('현재 모델의 양의 스케일:',scale)
print('각 행의 상대 점수 합:',relative.sum(dim=1))


### B3 · 대조 손실 해석
목표: 행·열의 정답과 후보 구별. 3×3 행렬에서 이미지 한 장의 정답 문장과 문장 하나의 정답 이미지는 어디에 있는가요? 후보를 늘리면 softmax는 그대로인가요? 힌트: 분모. 자가 점검: 대각선의 의미와 두 검색 방향을 설명합니다.


## 5. 문장으로 검색하고 top-k 검산하기
[`CLIPSearch.search`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L159)는 query와 k=3을 받아 rank·id·cosine·출처를 담은 표를 반환합니다.
이미지 인덱스를 먼저 준비해야 합니다. query만 새로 인코딩하고 점수를 내림차순으로 정렬합니다. 같은 점수에서는 원 사진 순서를 유지합니다.


In [ ]:
query='a cat in the snow'  # 눈 위의 고양이
result=lab.search(query,k=3)
display(result[['rank','id','cosine','category','source']])
scores=image_vectors@lab.encode_texts([query])[0]
manual_order=np.argsort(-scores.numpy(),kind='stable')[:3]
print('행렬곱으로 직접 검산한 ID:',gallery.iloc[manual_order].id.tolist())
fig,axes=plt.subplots(1,3,figsize=(10,3))
for ax,row in zip(axes,result.itertuples()):
    ax.imshow(Image.open(row.path));ax.set_title(f'{row.id}: {row.cosine:.3f}');ax.axis('off')
display(fig);plt.close(fig)


### B4 · top-k 빈칸
목표: 정렬 방향·원래 인덱스 대응. 설명용 점수 [0.2,0.9,0.4,0.7]의 상위2개 인덱스를 연습 변수에 적으세요. 오름차순 결과를 그대로 앞에서 두 개 고르면 왜 틀리는지 설명하세요. 힌트: 값이 큰 후보. 자가 점검: 첫 결과 점수가0.9입니다.


In [ ]:
my_top2=None  # 연습: 네 점수의 상위2개 원래 인덱스
if my_top2 is not None: print('내 인덱스:',my_top2)


## 6. Precision@k와 실패 사례
[`precision_at_k`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L53)는 관련 ID집합과 상위 결과로 비율을 계산합니다. 분모는 k이며, 중복 ID는 거절합니다.
눈 위 고양이의 사전 관련 집합은 P10,P11입니다. 말이 해변에 있는 사진은 이 모음에 없습니다.
관계 질의처럼 관련 이미지가1장뿐이면 모든 관련 이미지를 찾아도 Precision@3의 최대는1/3입니다. 이것만으로 검색이 실패했다고 해석하지 마세요.


In [ ]:
relevant={'P10','P11'}
print('눈 위 고양이 P@3:',precision_at_k(result.id.tolist(),relevant,3))
missing=lab.search('a horse on a beach',3)
display(missing[['id','cosine','category']])
print('모음에 없는 말 질의 P@3:',precision_at_k(missing.id.tolist(),set(),3))
rows=[]
for prompt in ['a photo of a dog','a red car','a cat playing with a lizard','눈 위의 고양이']:
    found=lab.search(prompt,3)
    rows.append({'query':prompt,'top3':', '.join(found.id),'top1_cosine':float(found.cosine.iloc[0])})
display(pd.DataFrame(rows))


### B5 · 검색 실패와 언어
목표: 반환값과 정답 구별. 없는 대상을 검색해도 왜3장이 나오는지, cosine=0.3을30% 정답 확률로 읽으면 안 되는 이유를 적으세요. 한국어와 영어 결과가 다르면 무엇을 결론낼 수 있나요? 힌트: 후보 중 순위와 사전학습 언어. 자가 점검: 이40장 실험의 범위를 넘지 않습니다.


## 7. 캐시를 다시 사용할 때
사진·모델 revision·라이브러리 버전·순서가 같으면 임베딩을 재사용합니다. 캐시 파일이 있다는 사실만으로 유효성을 인정하지 않고 shape·유한값·행 길이를 확인합니다.
실제 모델 출력이 아닌 임의 벡터를 성공 결과로 대신하지 않습니다.


In [ ]:
again=lab.index_images()
print('임베딩 캐시 재사용:',lab.cache_reused)
torch.testing.assert_close(again,image_vectors)


## 8. 마지막: AE/VAE와 CLIP을 같은 앱에서 관찰하기
A를 실행하지 않았어도 [`load_mnist`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/autoencoder.py#L92)와 [`load_checkpoints`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/representation.py#L190)로 자동 준비합니다.
[`build_embedding_app`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/embedding_app.py#L42)에 `search_lab=lab`을 주면 검색 탭도 연결됩니다. [`search_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/semantic_search.py#L176)는 `(query,k,lab)`을 받아 `(갤러리, 점수표, 상태문)`을 반환합니다.
[`reconstruction_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/embedding_app.py#L12)와 [`latent_view`](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/src/luna_genai/embedding_app.py#L28)는 앞선 복원·좌표 관찰을 재사용합니다. 함수의 입력·출력 순서와 컴포넌트 순서가 같아야 합니다.


In [ ]:
from luna_genai.autoencoder import load_mnist
from luna_genai.representation import load_checkpoints
from luna_genai.embedding_app import build_embedding_app,reconstruction_view,latent_view,APP_CSS
data=load_mnist(train_size=20000,validation_size=2000,test_size=1000)
models,provenance=load_checkpoints()
app=build_embedding_app(models,data.test,search_lab=lab)
gallery_output,score_output,message=search_view('a red car',3,lab)
display(score_output);print(message)


### B6 · 직접 조립: 두 문장 비교 탭
목표: 입력·callback·갤러리 이벤트 연결. 두 Textbox, 공통 k Slider, 비교 Button, 두 Gallery를 새 탭으로 만들고 두 질의 결과를 나란히 연결하세요. 기본 검색 탭은 독립적으로 동작합니다. 힌트: search_view는세 값을 반환하며 Gallery에는 첫 값을 줍니다. 자가 점검: dog와pizza를 넣으면 서로 다른 사진이 나옵니다.


In [ ]:
# 연습용 두 문장 비교 탭은 이 셀 위에 추가하세요.
app.launch(share=IN_COLAB,prevent_thread_lock=True,inline=IN_COLAB,quiet=True,css=APP_CSS)


In [ ]:
assert image_vectors.shape==(40,512)
assert torch.isfinite(image_vectors).all()
torch.testing.assert_close(image_vectors.norm(dim=1),torch.ones(40),atol=1e-5,rtol=1e-5)
assert similarities.argmax(dim=1).tolist()==[0,1,2]
assert result.id.tolist()==gallery.iloc[manual_order].id.tolist()
assert result.cosine.is_monotonic_decreasing
assert precision_at_k(result.id.tolist(),relevant,3)==2/3
assert precision_at_k(missing.id.tolist(),set(),3)==0
assert lab.cache_reused
assert len(gallery_output)==len(score_output)==3
assert all(Path(item[0]).is_file() for item in gallery_output)
assert np.isfinite(reconstruction_view('ae16',0,models,data.test)[3]['pixel_mse'])
assert latent_view('vae1',0.,0.,models)[0].shape==(28,28)
assert app.is_running
if not IN_COLAB: app.close()
print('W05A B runtime checks passed: 실제 CLIP·검색 검산·캐시·누적 앱 callback·서버 시작')


기본 검색 탭에서 문장과 k를 바꾸고 버튼을 누르세요. 결과의 사진·순위·점수·출처를 함께 확인합니다.
공유 URL은 영구 사이트가 아닙니다. 런타임 종료 후에는 준비 셀과 앱 셀을 다시 실행해야 합니다.
[강의노트](https://github.com/lunalab-ai/genAI/blob/2026-fall-w05a/course/notion/w05a/w05a-image-representations.md)와 기록지에서 복원·생성·정렬의 차이를 정리하세요.
